# `apptbl-llm-baseline`: off-the-shelf generative LLMs and decision models on the 200-respondent Vaccines subsample

In [7]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ASSET_NAME = "apptbl-llm-baseline"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
SRC_DIR = PAPER_DIR / "src"
if not (SRC_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from assets/; {SRC_DIR} has no microdata_analysis.py")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/apptbl-llm-baseline.*


## Labels and formatting (edit wording here)

In [8]:
SUMMARY_CSV = PAPER_DIR / "analysis" / "llm-baseline" / "results_summary.csv"
PERSONA = "Respondent"

# MSE header is two equations so Word can wrap it between them (one equation never wraps).
HEADERS = ["Model", "Reasoning effort", "Refusals", "Opt-out", r"$\mathrm{MSE}_\beta$ $\times 10^3$"]

# One table row each, in this order.
# (family, model, config, readout) select the row of results_summary.csv; config None means the model has one config.
# The last two entries are the text shown in the Model and Reasoning effort columns.
TABLE_ROWS = [
    ("Human benchmark", "Survey subsample (200 resp.)", "Observed choices", "Observed one-hot", "Human subsample (200 resp.)", ""),
    ("Generative LLM (Qwen)", "Qwen3.5-0.8B", "Thinking off", "Discrete text", "Qwen3.5-0.8B", "Off"),
    ("Generative LLM (Qwen)", "Qwen3.5-0.8B", "Thinking on", "Discrete text", "Qwen3.5-0.8B", "On"),
    ("Generative LLM (Qwen)", "Qwen3.5-4B", "Thinking off", "Discrete text", "Qwen3.5-4B", "Off"),
    ("Generative LLM (Qwen)", "Qwen3.5-4B", "Thinking on", "Discrete text", "Qwen3.5-4B", "On"),
    ("Generative LLM (Qwen)", "Qwen3.5-9B", "Thinking off", "Discrete text", "Qwen3.5-9B", "Off"),
    ("Generative LLM (Qwen)", "Qwen3.5-9B", "Thinking on", "Discrete text", "Qwen3.5-9B", "On"),
    ("Generative LLM (Gemini)", "Gemini 3.5 Flash-Lite", "Minimal thinking", "Discrete text", "Gemini 3.5 Flash-Lite", "Min"),
    ("Generative LLM (Gemini)", "Gemini 3.5 Flash-Lite", "High thinking", "Discrete text", "Gemini 3.5 Flash-Lite", "High"),
    ("Generative LLM (Gemini)", "Gemini 4 Argon", "Low thinking", "Discrete text", "Gemini 4 Argon", "Low"),
    ("Generative LLM (Gemini)", "Gemini 4 Argon", "High thinking", "Discrete text", "Gemini 4 Argon", "High"),
    ("Base decision model", "KEV-0.8B", None, "Continuous probs", "KEV-0.8B", ""),
    ("Base decision model", "KEV-4B", None, "Continuous probs", "KEV-4B", ""),
    ("Base decision model", "KEV-9B", None, "Continuous probs", "KEV-9B", ""),
]
COLUMN_RULES = [":---", ":---", "---:", "---:", "---:"]

## Build the table

In [9]:
import math
import re

from matplotlib import font_manager
from PIL import ImageFont

# Page and table style of manuscript/quarto-files/ms_template.docx. Word fixes the column widths that Pandoc
# derives from the rule dash counts, so a column narrower than its longest word splits that word across lines.
TEXT_WIDTH_PT = 468.0  # 8.5 in page minus 1 in left and right margins
TABLE_FONT_PT = 10.0  # Compact paragraph style used in table cells
CELL_PADDING_PT = 10.8  # Table style left + right cell margins (108 twips each)
COLUMN_SLACK_PT = 2.0  # per-column allowance for differences between Word and the font metrics measured here
MATH_WIDTH_FACTOR = 1.15  # Word sets $...$ in Cambria Math, which runs wider than Times New Roman
# No-break spaces around ×: Word sets each $...$ as one equation that never wraps.
MATH_SYMBOLS = {r"\beta": "β", r"\rho": "ρ", r"\times": "\u00a0×\u00a0"}


def load_table_font(weight):
    font_properties = font_manager.FontProperties(family="Times New Roman", weight=weight)
    font_path = font_manager.findfont(font_properties, fallback_to_default=False)
    return ImageFont.truetype(font_path, 1000)


REGULAR_FONT = load_table_font("normal")
BOLD_FONT = load_table_font("bold")


def shown_math(match):
    latex = match.group(1)
    text = latex
    for command, symbol in MATH_SYMBOLS.items():
        text = text.replace(command, symbol)
    text = text.replace(r"\mathrm", "")
    for mark in ("{", "}", "_", "^"):
        text = text.replace(mark, "")
    if "\\" in text:
        raise ValueError(f"No display text for a LaTeX command in {latex!r}; add it to MATH_SYMBOLS")
    return text


def shown_citation(match):
    surname = match.group(1).title()
    year = match.group(2)
    return f"({surname} {year})"


def shown_cell(cell):
    """Text Word displays for a markdown cell, with the font and width factor to measure it."""
    is_bold = cell.startswith("**") and cell.endswith("**")
    text = cell
    if is_bold:
        text = text[2:-2]
    text = re.sub(r"\[@([a-z]+)(\d{4})[a-z0-9]*\]", shown_citation, text)
    has_math = re.search(r"(?<!\\)\$", text) is not None
    text = re.sub(r"(?<!\\)\$(.+?)(?<!\\)\$", shown_math, text)
    text = text.replace(r"\$", "$")
    text = text.replace(r"\|", "|")
    font = BOLD_FONT if is_bold else REGULAR_FONT
    factor = MATH_WIDTH_FACTOR if has_math else 1.0
    return text, font, factor


def text_width_pt(text, font, factor):
    return font.getlength(text) / 1000.0 * TABLE_FONT_PT * factor


def unbreakable_pieces(text):
    """Pieces Word keeps on one line: words, with compound words (letter-hyphen-letter) split after the hyphen."""
    pieces = []
    for word in re.split(r"[ \t\n]+", text):
        pieces.extend(re.split(r"(?<=[A-Za-z]-)(?=[A-Za-z])", word))
    return pieces


def column_widths_pt(headers, rows):
    """Each column gets at least its widest unbreakable piece; leftover page width goes to columns that wrap."""
    n_columns = len(headers)
    min_pt = [0.0] * n_columns
    natural_pt = [0.0] * n_columns
    for cells in [headers, *rows]:
        if len(cells) != n_columns:
            raise ValueError(f"Row has {len(cells)} cells, expected {n_columns}: {cells}")
        for column_index, cell in enumerate(cells):
            text, font, factor = shown_cell(cell)
            for piece in unbreakable_pieces(text):
                piece_pt = text_width_pt(piece, font, factor)
                min_pt[column_index] = max(min_pt[column_index], piece_pt)
            cell_pt = text_width_pt(text, font, factor)
            natural_pt[column_index] = max(natural_pt[column_index], cell_pt)
    extra_pt = CELL_PADDING_PT + COLUMN_SLACK_PT
    min_pt = [width + extra_pt for width in min_pt]
    natural_pt = [width + extra_pt for width in natural_pt]
    if sum(min_pt) > TEXT_WIDTH_PT:
        raise ValueError(
            f"Longest words need {sum(min_pt):.0f}pt but the page is {TEXT_WIDTH_PT:.0f}pt wide; "
            f"shorten labels or drop columns. Per-column minimum: {[round(width) for width in min_pt]}"
        )
    if sum(natural_pt) <= TEXT_WIDTH_PT:
        return natural_pt
    spare_pt = TEXT_WIDTH_PT - sum(min_pt)
    stretch_pt = [natural - minimum for natural, minimum in zip(natural_pt, min_pt)]
    total_stretch_pt = sum(stretch_pt)
    widths_pt = []
    for minimum, stretch in zip(min_pt, stretch_pt):
        widths_pt.append(minimum + spare_pt * stretch / total_stretch_pt)
    return widths_pt


def escape_markdown(text):
    """Escape $ and | so Pandoc reads them as text inside a pipe table."""
    escaped = text.replace("$", r"\$")
    escaped = escaped.replace("|", r"\|")
    return escaped


def pipe_table(headers, rows, column_rules):
    """Pandoc pipe table. column_rules set alignment (':---' left, '---:' right); dash counts come from column_widths_pt."""
    if len(column_rules) != len(headers):
        raise ValueError(f"{len(column_rules)} column rules for {len(headers)} columns")
    widths_pt = column_widths_pt(headers, rows)
    rule_cells = []
    for column_rule, width_pt in zip(column_rules, widths_pt):
        if re.fullmatch(r":?-+:?", column_rule) is None:
            raise ValueError(f"Bad column rule {column_rule!r}")
        dashes = "-" * round(width_pt)
        left = ":" if column_rule.startswith(":") else ""
        right = ":" if column_rule.endswith(":") else ""
        rule_cells.append(left + dashes + right)
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join(rule_cells) + "|")
    for row in rows:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines) + "\n"

In [10]:
summary_df = pd.read_csv(SUMMARY_CSV)

rows = []
for family, model, config, readout, shown_model, shown_effort in TABLE_ROWS:
    mask = (summary_df["family"] == family) & (summary_df["model"] == model)
    mask = mask & (summary_df["persona"] == PERSONA) & (summary_df["readout"] == readout)
    if config is not None:
        mask = mask & (summary_df["config"] == config)
    matches = summary_df[mask]
    if len(matches) != 1:
        raise ValueError(f"Expected 1 row in {SUMMARY_CSV} for {model}/{config}, got {len(matches)}")
    result = matches.iloc[0]
    rows.append([
        shown_model,
        shown_effort,
        f"{int(result['refusals'])}",
        f"{result['opt_out_pct']:.1f}%",
        f"{result['beta_mse_x1e3']:.2f}",
    ])

table_md = pipe_table(HEADERS, rows, COLUMN_RULES)
display(pd.DataFrame(rows, columns=HEADERS))

,Model,Reasoning effort,Refusals,Opt-out,$\mathrm{MSE}_\beta$ $\times 10^3$
0,Human subsample (200 resp.),,0,20.3%,0.46
1,Qwen3.5-0.8B,Off,0,59.0%,2.75
2,Qwen3.5-0.8B,On,19,28.6%,13.82
3,Qwen3.5-4B,Off,0,53.6%,1.94
4,Qwen3.5-4B,On,0,4.8%,9.71
5,Qwen3.5-9B,Off,0,81.8%,1.49
6,Qwen3.5-9B,On,1,11.4%,6.27
7,Gemini 3.5 Flash-Lite,Min,0,21.9%,10.64
8,Gemini 3.5 Flash-Lite,High,0,4.8%,12.28
9,Gemini 4 Argon,Low,0,12.2%,16.50


## Save `apptbl-llm-baseline.md` (included by the manuscript)

In [11]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

| Model | Reasoning effort | Refusals | Opt-out | $\mathrm{MSE}_\beta$ $\times 10^3$ |
|:-------------------------------------------------------------------------------------------------------------------------------------|:--------------------------------------------------------------------------------|-----------------------------------------------:|--------------------------------------------:|------------------------------------------------------------------------------:|
| Human subsample (200 resp.) |  | 0 | 20.3% | 0.46 |
| Qwen3.5-0.8B | Off | 0 | 59.0% | 2.75 |
| Qwen3.5-0.8B | On | 19 | 28.6% | 13.82 |
| Qwen3.5-4B | Off | 0 | 53.6% | 1.94 |
| Qwen3.5-4B | On | 0 | 4.8% | 9.71 |
| Qwen3.5-9B | Off | 0 | 81.8% | 1.49 |
| Qwen3.5-9B | On | 1 | 11.4% | 6.27 |
| Gemini 3.5 Flash-Lite | Min | 0 | 21.9% | 10.64 |
| Gemini 3.5 Flash-Lite | High | 0 | 4.8% | 12.28 |
| Gemini 4 Argon | Low | 0 | 12.2% | 16.50 |
| Gemini 4 Argon | High | 0 | 9.1% | 16.57 |
| KEV-0.8B |  | 0 | 32.1% | 5